** Gerei a chave usando o Google AI Studio e rodei dentro do proprio ambiente**

Rodei o codigo dentro do google colab, sendo assim a chave da API eu passei dentro da parte de "Secrets" que fica na propria extensão do colab se for for rodar local tem que passar a chave no proprio codigo

In [8]:
!pip install google-genai pydantic

In [5]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
print("Chave API configurada no ambiente.")

Chave API configurada no ambiente.


In [7]:
import os
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

class SentimentAnalysis(BaseModel):
    """Estrutura para análise de sentimento."""
    polaridade: str = Field(description="O sentimento principal do texto: 'Positivo', 'Neutro' ou 'Negativo'.")
    score: float = Field(description="Uma pontuação de 0.0 a 1.0 indicando a confiança na polaridade.")

def analisar_sentimento_llm(texto_input: str) -> dict:
    """
    Usa o modelo Gemini para analisar o sentimento de um texto
    e retornar a polaridade e o score em formato JSON.
    """

    try:
        if not os.getenv("GEMINI_API_KEY"):
             raise ValueError("Variável de ambiente GEMINI_API_KEY não está configurada.")

        client = genai.Client()
    except Exception as e:
        return {"erro": "Falha na inicialização do cliente LLM. Verifique sua chave API."}

    prompt = f"""
    Analise o seguinte texto e determine o sentimento.
    Classifique a 'polaridade' como 'Positivo', 'Neutro' ou 'Negativo'.
    Estime o 'score' de confiança na análise entre 0.0 e 1.0.

    TEXTO A SER ANALISADO:
    ---
    {texto_input}
    ---
    """

    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=SentimentAnalysis,
    )

    print("-> Enviando texto para análise...")
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=config,
        )

        resultado_json = response.text
        return json.loads(resultado_json)

    except Exception as e:
        print(f"Ocorreu um erro durante a chamada da API: {e}")
        return {"erro": str(e)}

if __name__ == "__main__":

    print("--------------------------------------------------")
    texto_input = input("Digite o texto para Análise de Sentimento: \n> ")
    print("--------------------------------------------------")

    if not texto_input.strip():
        print("Nenhum texto fornecido. Encerrando.")
    else:
        resultado_analise = analisar_sentimento_llm(texto_input)

        print("\n" + "="*50)
        print("✨ RESULTADO DA ANÁLISE DE SENTIMENTO ✨")
        print("="*50)
        print(f"TEXTO ORIGINAL: {texto_input}")
        print("-" * 50)

        if "erro" in resultado_analise:
            print(f"STATUS: FALHA\n{resultado_analise['erro']}")
        else:
            print(f"STATUS: SUCESSO")
            print(f"Polaridade: {resultado_analise.get('polaridade')}")
            print(f"Score: {resultado_analise.get('score'):.2f}")
            print("-" * 50)
            print("Estrutura JSON Completa:")
            print(json.dumps(resultado_analise, indent=2, ensure_ascii=False))

--------------------------------------------------
Digite o texto para Análise de Sentimento: 
> odeio sorvete
--------------------------------------------------
-> Enviando texto para análise...

✨ RESULTADO DA ANÁLISE DE SENTIMENTO ✨
TEXTO ORIGINAL: odeio sorvete
--------------------------------------------------
STATUS: SUCESSO
Polaridade: Negativo
Score: 0.95
--------------------------------------------------
Estrutura JSON Completa:
{
  "polaridade": "Negativo",
  "score": 0.95
}
